In [ ]:
!pip install -qU langchain langchain-openai langchain-community langchain-experimental neo4j tiktoken ragas

In [ ]:
import os
import getpass
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# API 키 설정 (필요 시)
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

In [ ]:
print("=== [연구자용 RAG] 내부 논문 DB (Vector) 초기화 ===")
# 대학원 연구실의 내부(과거) 데이터베이스라고 가정합니다.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = InMemoryVectorStore(embeddings)
vectorstore.add_texts([
    "[내부 DB - 2023년 보고서] 신약 'NovaX'의 임상 2상 성공률은 45%로 기록되었다.",
    "[내부 DB - 2022년 보고서] 'NovaX'의 주요 부작용은 가벼운 두통이다."
])
internal_retriever = vectorstore.as_retriever()

@tool
def search_internal_papers(query: str) -> str:
    """연구실 내부 Vector DB를 검색합니다. 과거의 연구 기록이나 임상 데이터를 찾을 때 사용하세요."""
    docs = internal_retriever.invoke(query)
    return "\n".join([doc.page_content for doc in docs])

In [ ]:
# ==========================================
# 👩‍💻 [개별 실습 과제 (TODO) - 권장 시간: 20분]
# ==========================================
print("\n=== 📝 TODO: 외부 Web Search 연동 및 교차 검증 에이전트 구축 ===")
print("연구 상황: 내부 DB는 2023년까지의 데이터만 있습니다.")
print("에이전트가 내부 DB를 먼저 찾고, 정보가 부족하거나 최신 정보가 필요하면 외부 검색(Mock)을 수행한 뒤,")
print("수치 연산 도구를 이용해 최종 통계값을 계산하도록 만들어야 합니다.")

# TODO 1: 외부 검색 도구(Mock) 생성하기
# 실제 Web Search API(Tavily 등) 대신, 특정 키워드가 들어오면 최신 정보를 반환하는 도구를 만드세요.
# 조건: 'NovaX'와 '3상' 혹은 '최신' 이라는 키워드가 들어오면 "[웹 검색 - 2026년 뉴스] NovaX 임상 3상 완료, 최종 성공률 68% 달성, 투여 환자 수 5000명" 을 반환하게 하세요.
# @tool
# def search_external_web(query: str) -> str:
#     """최신 논문이나 외부 뉴스를 검색할 때 사용합니다."""
#     # 로직 구현 (if문 활용)
#     pass

# TODO 2: 임상 데이터 통계 계산 도구 생성 (Type-Hinting 엄격 적용)
# 환자 수(int)와 성공률(float, 예: 0.68)을 입력받아 완치된 예상 환자 수를 계산(int 변환)하는 도구를 만드세요.
# @tool
# def calculate_cured_patients(patient_count: int, success_rate: float) -> int:
#     """[Docstring을 정교하게 작성하여 에이전트가 파라미터 타입을 헷갈리지 않게 하세요]"""
#     # 로직 구현
#     pass

# TODO 3: 에이전트 시스템 프롬프트(state_modifier) 작성 및 에이전트 생성
# [프롬프트 필수 조건]
# 1. 반드시 search_internal_papers를 먼저 사용할 것.
# 2. 내부 데이터가 2024년 이전 것이라면 반드시 search_external_web을 통해 최신 데이터를 교차 검증할 것.
# 3. 계산이 필요하면 반드시 calculate_cured_patients 도구를 사용할 것.

# tools = [search_internal_papers, ???, ???]
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# system_prompt = """[여기에 정교한 지시사항을 작성하세요]"""
# agent_executor = create_react_agent(llm, tools, prompt=system_prompt)

# TODO 4: 복합 추론 질문 실행
# 질문: "우리 연구실에 기록된 NovaX의 임상 성공률을 확인해보고, 최신 웹 검색을 통해 3상 결과가 나왔는지 교차 검증해줘. 그리고 3상 결과의 환자수와 성공률을 바탕으로 완치된 예상 환자 수를 계산해줘."
# res = agent_executor.invoke({"messages": [("user", "???")]})
# print(res['messages'][-1].content)